# 10.1 General principles of data correspondence

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

Consider the following declarations from diet.mod in [Chapter 2](../02/02.md), defining the set
FOOD and three parameters indexed over it:

```ampl
set FOOD;
param cost {FOOD} > 0;
param f_min {FOOD} >= 0;
param f_max {j in FOOD} >= f_min[j];
```

A relational table giving values for these components has four columns:

```ampl
FOOD   cost     f_min   f_max
BEEF   3.19     2       10
CHK    2.59     2       10
FISH   2.29     2       10
HAM    2.89     2       10
MCH    1.89     2       10
MTL    1.99     2       10
SPG    1.99     2       10
TUR    2.49     2       10
```

The column headed FOOD lists the members of the AMPL set also named FOOD. This is
the table's key column; entries in a key column must be unique, like a set's members, so
that each key value identifies exactly one row. The column headed cost gives the values
of the like-named parameter indexed over set FOOD; here the value of
`cost["BEEF"]` is specified as 3.19, `cost["CHK"]` as 2.59, and so forth. The
remaining two columns give values for the other two parameters indexed over FOOD.

The table has eight rows of data, one for each set member. Thus each row contains all
of the table's data corresponding to one member — one food, in this example.

In the context of database software, the table rows are often viewed as data records,
and the columns as fields within each record. Thus a data entry form has one entry field
for each column. A form for the diet example (from Microsoft Access) might look like
[Figure 10-1](#fig-10-1). Data records, one for each table row, can be entered or viewed one at a time
by using the controls at the bottom of the form.

**[Figure 10-1](#fig-10-1):** Access data entry form.

Parameters are not the only entities indexed over the set FOOD in this example. There
are also the variables:

```ampl
var Buy {j in FOOD} >= f_min[j], <= f_max[j];
```

and assorted result expressions that may be displayed:

```ampl
ampl: model diet.mod;
ampl: data diet2a.dat;
ampl: solve;
MINOS 5.5: optimal solution found.
13 iterations, objective 118.0594032
ampl: display Buy, Buy.rc, {j in FOOD} Buy[j]/f_max[j];
:        Buy         Buy.rc    Buy[j]/f_max[j]    :=
BEEF    5.36061    8.88178e-16     0.536061
CHK     2          1.18884         0.2
FISH    2          1.14441         0.2
HAM    10         -0.302651        1
MCH    10         -0.551151        1
MTL    10         -1.3289          1
SPG     9.30605    0               0.930605
TUR     2          2.73162         0.2
;
```

All of these can be included in the relational table for values indexed over FOOD:

```ampl
FOOD  cost  f_min  f_max    Buy        BuyRC       BuyFrac
BEEF  3.19  2      10     5.36061    8.88178e-16   0.536061
CHK   2.59  2      10     2          1.18884       0.2
FISH  2.29  2      10     2          1.14441       0.2
HAM   2.89  2      10    10         -0.302651      1
MCH   1.89  2      10    10         -0.551151      1
MTL   1.99  2      10    10         -1.3289        1
SPG   1.99  2      10     9.30605    0             0.930605
TUR   2.49  2      10     2          2.73162       0.2
```

Where the first four columns would typically be read into AMPL from a database, the last
three are results that would be written back from AMPL to the database. We have
invented the column headings `BuyRC` and `BuyFrac`, because the AMPL expressions for
the quantities in those columns are typically not valid column headings in database management
systems. The table declaration provides for input/output and naming distinctions
such as these, as subsequent sections will show.

Other entities of diet.mod are indexed over the set NUTR of nutrients: parameters
n_min and n_max, dual prices and other values associated with constraint Diet, and
expressions involving these. Since nutrients are entirely distinct from foods, however,
the values indexed over nutrients go into a separate relational table from the one for
foods. It might look like this:

```ampl
NUTR    n_min   n_max   NutrDual
A         700   20000    0
B1        700   20000    0
B2        700   20000    0.404585
C         700   20000    0
CAL     16000   24000    0
NA          0   50000   -0.00306905
```

As this example suggests, any model having more than one indexing set will require more
than one relational table to hold its data and results. Databases that consist of multiple
tables are a standard feature of relational data management, to be found in all but the simplest
"flat file" database packages.

Entities indexed over the same higher-dimensional set have a similar correspondence
to a relational table, but with one key column for each dimension. In the case of [Chapter 4](../04/04.md)'s
steelT.mod, for example, the following parameters and variables are indexed over
the same two-dimensional set of product-time pairs:

```ampl
set PROD;          # products
param T > 0;       # number of weeks
param market {PROD,1..T} >= 0;
param revenue {PROD,1..T} >= 0;
var Make {PROD,1..T} >= 0;
var Sell {p in PROD, t in 1..T} >= 0, <= market[p,t];
```

A corresponding relational table thus has two key columns, one containing members of
PROD and the other members of 1..T, and then a column of values for each parameter
and variable. Here's an example, corresponding to the data in steelT.dat:

```ampl
PROD   TIME  market revenue  Make    Sell
bands   1     6000     25    5990    6000
bands   2     6000     26    6000    6000
bands   3     4000     27    1400    1400
bands   4     6500     27    2000    2000
coils   1     4000     30    1407     307
coils   2     2500     35    1400    2500
coils   3     3500     37    3500    3500
coils   4     4200     39    4200    4200
```

Each ordered pair of items in the two key columns is unique in this table, just as these
pairs are unique in the set `{PROD,1..T}`. The market column of the table implies,
for example, that `market["bands",1]` is 6000 and that `market["coils",3]` is 3500.
From the first row, we can also see that `revenue["bands",1]` is 25,
`Make["bands",1]` is 5990, and `Sell["bands",1]` is 6000. Again various names
from the AMPL model are used as column headings, except for `TIME`, which must be
invented to stand for the expression `1..T`. As in the previous example, the column headings
can be any identifiers acceptable to the database software, and the table declaration
will take care of the correspondences to AMPL names (as explained below).
AMPL entities that have sufficiently similar indexing generally fit into the same relational
table. We could extend the steelT.mod table, for instance, by adding a column
for values of

```ampl
var Inv {PROD,0..T} >= 0;
```

The table would then have the following layout:

```ampl
PROD     TIME  market  revenue   Make     Sell    Inv
bands     0        .       .        .        .     10
bands     1     6000      25     5990     6000      0
bands     2     6000      26     6000     6000      0
bands     3     4000      27     1400     1400      0
bands     4     6500      27     2000     2000      0
coils     0        .       .        .        .      0
coils     1     4000      30     1407      307   1100
coils     2     2500      35     1400     2500      0
coils     3     3500      37     3500     3500      0
coils     4     4200      39     4200     4200      0
```

We use "." here to mark table entries that correspond to values not defined by the
model and data. There is no `market["bands",0]` in the data for this model, for
example, although there does exist a value for `Inv["bands",0]` in the results. Database
packages vary in their handling of "missing" entries of this sort.

Parameters and variables may also be indexed over a set of pairs that is read as data
rather than being constructed from one-dimensional sets. For instance, in the example of
transp3.mod from [Chapter 3](../03/03.md), we have:

```ampl
set LINKS within {ORIG,DEST};
param cost {LINKS} >= 0;   # shipment costs per unit
var Trans {LINKS} >= 0;    # actual units to be shipped
```

A corresponding relational table has two key columns corresponding to the two components
of the indexing set LINKS, plus a column each for the parameter and variable that
are indexed over LINKS:

```ampl
ORIG    DEST   cost  Trans
GARY     DET    14       0
GARY     LAF     8     600
GARY     LAN    11       0
GARY     STL    16     800
CLEV     DET     9    1200
CLEV     FRA    27       0
CLEV     LAF    17     400
CLEV     LAN    12     600
CLEV     STL    26       0
CLEV     WIN     9     400
PITT     FRA    24     900
PITT     FRE    99    1100
PITT     STL    28     900
PITT     WIN    13       0
```

The structure here is the same as in the previous example. There is a row in the table
only for each origin-destination pair that is actually in the set LINKS, however, rather
than for every possible origin-destination pair.